# Scrapping paso a paso de concellos


## Paso 1 . Comprobar descarga

### Pasos para comprobar con Robotfileparser 
1. Realizar una consulta get para recoger el contenido del archivo txt a tratar con las cabeceras que hemos definido previamente 
    * Hacemos una comprobación previa para comprobar que la url a tratar existe , esto dentro de un try catch 
2. Utilizamos RobotFileParser para interpretar las reglas que contiene el archivo txt
3. Debido a la existencia de líneas BOM(indicando codificación) en el txt , procedemos a su eliminación para una lectura más limpia y que no interfiera en la interpretación de las líneas del archivo.
4. Buscamos meidante un bucle for si la url a analizar de concellos permite a nuestro User-Agent realizar un scrapping, esto se realiza por un can_fetch 
    *  Comprueba si la URL coincide con alguna ruta prohibida robots

In [142]:
import time
from io import StringIO
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE = "https://en.wikipedia.org/"
URL_municipios = urljoin(BASE,"wiki/List_of_municipalities_in_A_Coru%C3%B1a")
CABECEIRAS = {"User-Agent": "E4_Concellos (carolinamesacurso@gmail.com)"}
PAUSA = 1.0                 

RAW = Path("../datos/raw")
CLEAN = Path("../datos/clean")
RAW.mkdir(parents=True, exist_ok=True)
CLEAN.mkdir(parents=True, exist_ok=True)

In [143]:
respuesta_robots = requests.get(urljoin(BASE, "robots.txt"),headers=CABECEIRAS,timeout=10)

print("Código:", respuesta_robots.status_code)
try:
    respuesta_robots.raise_for_status()
except requests.HTTPError as e:
    print("HTTPError:", e)

# print(respuesta_robots.text)

robots = RobotFileParser(urljoin(BASE, "robots.txt"))

#mostrar caracteres especiales
print(repr(respuesta_robots.text[:20]))

# Eliminamos el BOM y cargamos las reglas
robots.parse(respuesta_robots.text.lstrip("\ufeff").splitlines())

# Comprobamos si podemos acceder a la página

for ruta in ["wiki/List_of_municipalities_in_A_Coru%C3%B1a"]:
    print(f"{ruta} permitido? "f"{robots.can_fetch(CABECEIRAS['User-Agent'], urljoin(BASE, ruta))}")

# Consultamos el tiempo de espera
print("Crawl-delay:", robots.crawl_delay(CABECEIRAS["User-Agent"]))

Código: 200
'\ufeff# robots.txt for ht'
wiki/List_of_municipalities_in_A_Coru%C3%B1a permitido? True
Crawl-delay: None


    
### Que permite la licencia y datos personales que existen en ella
* La licencia que nos permite leer las páginas reglamentarias de wikipedia se llama  'CC BY-SA 4.0.', la cual nos permite modificar , leer , copiar y reutilizar el contenido .
* Podemos observar por los datos mostrados anteriormente que tenemos ciertos agentes que no están permitidos y ciertas páginas las cuales no podemos scrapear.
* En el caso de la página de concellos vemos que no existen datos personales ya que solo observamso datos de los concellos mostrados públicamente.

Disallow: /w/
Disallow: /api/
Disallow: /trap/
Disallow: /wiki/Special:

# Paso 2 . Descarga

### Realizar comprobaciones adicionales
* Comprobamos que la codificación de la url a tratar es válida o la indicada para trabajar . En este caso comprobamos que es utf-8 asi que estaría correcto.

In [144]:
respuesta = requests.get(URL_municipios, headers=CABECEIRAS, timeout=10)
print("Código de estado:", respuesta.status_code)
print("Content-Type    :", respuesta.headers.get("Content-Type"))
print("Codificación    :", respuesta.encoding)
print()
print(respuesta.text[:600])

Código de estado: 200
Content-Type    : text/html; charset=UTF-8
Codificación    : UTF-8

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard skin-un


* Procedemos a la descarga de esta url con nuestro propio UserAgent antes declarado 
* Si la ruta (RAW / "municipios_coruna.html") ya existía no se descargará dos veces y mostrará un mensaje parando así la función

In [145]:
def descargar(url, destino):
    if destino.exists():
        return "ya existía, se cancela la descarga"

    r = requests.get(url, headers=CABECEIRAS, timeout=10)
    r.raise_for_status()
    destino.write_bytes(r.content)
    time.sleep(PAUSA)

    return "descargado"

fichero = RAW / "municipios_coruna.html"

# Descargamos la página
print(fichero.name, descargar(URL_municipios, fichero))

municipios_coruna.html descargado


## Paso 3 . Extracción

* Al investigar en la propia web encontramos que la tabla que queremos descargar se encuentra con la siguiente estructura :
* Analizando esta estructura descubrimos que el contenido de la tabla se basa en una tabla con un body, el cual contiene entre otras etiquetas tr ,td y th . Para recoger la tabla indicada tendremos que acceder a sus cabeceras que están dentro de etiquetas th y comprobar su contenido
* Para ello recogemos todas las tablas ,recorriendo en ellas todas las etiquetas th de las tablas existentes hasta que alguna coincida con el texto 'Population(2024)'
<small>

```html
<table class="wikitable sortable" id="mwvQ">
<tbody id="mwvg"><tr id="mwvw"><th id="mwwA"><b id="mwwQ">Name</b></th>
<th id="mwwg"><b id="mwww">Population (2024)<sup about="#mwt79" class="mw-ref reference" data-mw='{"name":"ref","attrs":{},"body":{"id":"mw-reference-text-cite_note-20"}}' id="cite_ref-20" rel="dc:references" typeof="mw:Extension/ref"><a href="#cite_note-20" id="mwxA"><span class="mw-reflink-text" id="mwxQ"><span class="cite-bracket" id="mwxg">[</span>20<span class="cite-bracket" id="mwxw">]</span></span></a></sup></b></th></tr>
<tr id="mwyA">
<td id="mwyQ"><a href="https://en.wikipedia.org/wiki/Abegondo" id="mwyg" rel="mw:WikiLink" title="Abegondo">Abegondo</a></td>
<td id="mwyw">5,578</td></tr>
<tr id="mwzA">
...
```

</small>

In [146]:
sopa = BeautifulSoup(fichero.read_bytes(), "html.parser") 
# Buscamos todas las tablas
tablas = sopa.find_all("table")

tabla_municipios = None

for tabla in tablas:
 
    cabeceras = [
        th.get_text(" ", strip=True)
        for th in tabla.find_all("th")
    ]
    for cabecera in cabeceras:
        if "Population (2024)" in cabecera:
            tabla_municipios = tabla
            break
    if tabla_municipios is not None:
        break

print("Tabla encontrada:", tabla_municipios is not None)
# print(tabla_municipios)


Tabla encontrada: True


* Procedemos en la misma función que extraemos la tabla a convertir todos los td en columnas de nuestro nuevo dataframe
* Para esto recorremos los tr y extraemos los td , para convertir el primer td como concello y el segundo como población

In [147]:
def extraer_municipios(fichero):

    sopa = BeautifulSoup(fichero.read_bytes(), "html.parser")

    tablas = sopa.find_all("table")
    tabla_municipios = None

    for tabla in tablas:
        cabeceras = [
            th.get_text(" ", strip=True)
            for th in tabla.find_all("th")
        ]

        for cabecera in cabeceras:
            if "Population (2024)" in cabecera:
                tabla_municipios = tabla
                break

        if tabla_municipios is not None:
            break

    filas = []

    for tr in tabla_municipios.find_all("tr"):
        celdas = tr.find_all("td")

        if len(celdas) == 2:
            filas.append({
                "municipio": celdas[0].get_text(strip=True),
                "poblacion": celdas[1].get_text(strip=True)
            })

    return filas

filas = extraer_municipios(fichero)

municipios = pd.DataFrame(filas)

print(municipios)


     municipio poblacion
0     Abegondo     5,578
1         Ames    32,812
2       Aranga     1,799
3         Ares     6,156
4      Arteixo    34,038
..         ...       ...
88       Vedra     4,942
89  Vilarmaior     1,252
90  Vilasantar     1,229
91    Vimianzo     6,808
92         Zas     4,261

[93 rows x 2 columns]


## Paso 4. Limpieza

* Comprobamos que si todas las poblaciones anteriormente extraidas tiene el mismo formato , es decir , si un campo que representa una cantidad numérica lleva caractéres especiales como '.' o ',' para indicar separadores de miles.
* En este caso mostramos que existen cantidades las cuales llevan ',' 

In [148]:
print("Valores con coma:", municipios["poblacion"].str.contains(",").sum())
print("Valores con punto:", municipios["poblacion"].str.contains(r"\.").sum())


Valores con coma: 93
Valores con punto: 0


In [149]:
print(municipios["poblacion"].value_counts().head(10))

poblacion
5,578     1
32,812    1
1,799     1
6,156     1
34,038    1
5,887     1
3,291     1
6,986     1
13,261    1
1,801     1
Name: count, dtype: int64


* Procedemos a convertir los valores de la columna de población a dato numérico , mostrando cuales no se han podido convertir .
* En el caso de que existiesen datos que no se hayan podido convertir procederemos a gestionarlos con la etiqueta de la librería pandas 'coerce', el cual sustituye los valores incorrectos en valores nulos .
* También se podría investigar de analizar que parámetros nos está interfiriendo en el tratamiento de estos números para sustituir este convirtiendolo en miles, para ello lo guardaríamos en una variable en el proceso.

In [150]:
def numero_es(serie):
    return pd.to_numeric(
        serie.str.replace(",", "", regex=False).str.replace(".", "", regex=False).astype(int),
        errors="coerce",
    )

for col in ["poblacion"]:
    municipios[col] = numero_es(municipios[col])
print(municipios.dtypes)
print("Municipios sin población:", municipios["poblacion"].isna().sum())


municipio      str
poblacion    int64
dtype: object
Municipios sin población: 0


In [151]:
print(municipios.head(10))


  municipio  poblacion
0  Abegondo       5578
1      Ames      32812
2    Aranga       1799
3      Ares       6156
4   Arteixo      34038
5     Arzúa       5887
6    A Baña       3291
7  Bergondo       6986
8  Betanzos      13261
9  Boimorto       1801


## Paso 5. Validación

* Comprobamos que al mirar las estadísticas numéricas del nuevo dataframe , nos marca que tenemos 93 concellos y 93 concellos que tienen valores únicos .
* Comprobando así que no hay concellos duplicados , lo confirmamos cuando mostramos los valores duplicados en forma de suma

In [152]:
municipios.describe(include="all")

,municipio,poblacion
count,93,93.000000
unique,93,NaN
top,Abegondo,NaN
freq,1,NaN
mean,NaN,12132.473118
std,NaN,28528.028535
min,NaN,1004.000000
25%,NaN,2979.000000
50%,NaN,5195.000000
75%,NaN,9064.000000


In [153]:
print("Número de municipios:", len(municipios))
print("Municipios duplicados:", municipios["municipio"].duplicated().sum())

Número de municipios: 93
Municipios duplicados: 0


* Para poder comparar la suma de la población en nuestra tabla con la población mostrada en la página web procedemos a los siguientes pasos :
    * Sumamos toda la columna de nuestro dataframe llamada población 
    * En la página web recorremos todos los párrafos para extraer el texto que nos interesa , este identificado con la palabra 'inhabitants'
    * Despues de guardar en una variable ese texto extraemos el dato y remplazamos las comas o puntos por espacios en blanco para mejorar su lectura y realizar su posterior comparación , donde vemos que son el msimo número 

In [154]:
municipios["poblacion"].sum()
#A partir de 2024 Según el censo español, la provincia es la duodécima más grande por población , con 1.128.320 habitantes

np.int64(1128320)

In [155]:
texto_habitantes = ""

parrafos = sopa.find_all("p")

for parrafo in parrafos:
    texto = parrafo.get_text(" ", strip=True)

    if "inhabitants" in texto.lower():
        texto_habitantes = texto
        break

print(texto_habitantes)

A Coruña is a province in the autonomous community of Galicia , Spain, that is divided into 93 municipalities . As of the 2024 [ update ] Spanish census, the province is the 12th largest by population , with 1,128,320 inhabitants, [ 1 ] and the 32nd largest by land area, spanning 7,950 square kilometres (3,070 sq mi) . [ 2 ] Municipalities are the basic local political division in Spain [ 3 ] and can only belong to one province. [ 4 ] They enjoy a large degree of autonomy in their local administration, being in charge of tasks such as urban planning, water supply, lighting, roads, local police, and firefighting. [ 5 ]


In [156]:
poblacion_provincia = int(texto_habitantes.split("inhabitants")[0].split()[-1].replace(",", "").replace(".", ""))

print(poblacion_provincia)

1128320


In [157]:
print(poblacion_provincia == municipios["poblacion"].sum())

True


* Al recorrer la tabla con read_html observamos las siguientes características , tanto de resultados como de procedimiento a realizar los anteriores resultados :
    * Observamos que para crear la columna 'población' a datos numéricos , esto se tiene que realizar en varios pasos ya que al poder indicar solo un caracter como decimal no recorre todos los formatos que se presentan en la tabla
    * Cuando vamos a proceder con la comprobación de los datos duplicados en la columna concellos, observamos que no nos muestra las estadísticas numéricas perdiendo así información importante
    * Pero llegamos a los mismos resultados numéricos que con la librería pandas

In [158]:
tabla_readhtml = pd.read_html(StringIO(fichero.read_text(encoding="utf-8")),decimal=",",thousands=".", na_values=["-"]) #decimal=".", thousands=",",
print("Tablas encontradas:", len(tabla_readhtml))
tabla_readhtml[0].head()

Tablas encontradas: 2


,Name,Population (2024)[20]
0,Abegondo,5.578
1,Ames,32.812
2,Aranga,1.799
3,Ares,6.156
4,Arteixo,34.038


In [159]:
tabla_readhtml = pd.read_html(
    StringIO(fichero.read_text(encoding="utf-8")),
    converters={1: str},
    na_values=["—"]
)

municipios_readhtml = tabla_readhtml[0]

In [160]:
municipios_readhtml["Population (2024)[20]"] = (
    municipios_readhtml["Population (2024)[20]"]
    .str.replace(",", "", regex=False)
    .str.replace(".", "", regex=False)
)

In [161]:
municipios_readhtml["Population (2024)[20]"] = (
    pd.to_numeric(
        municipios_readhtml["Population (2024)[20]"],
        errors="coerce"
    ).astype("Int64")
)

In [162]:
municipios_readhtml["Population (2024)[20]"].head(20).tolist()

[5578,
 32812,
 1799,
 6156,
 34038,
 5887,
 3291,
 6986,
 13261,
 1801,
 19018,
 4200,
 8197,
 4098,
 3296,
 5098,
 24781,
 1182,
 31595,
 3691]

In [163]:
municipios_readhtml.describe(include="all")

,Name,Population (2024)[20]
count,93,93.0
unique,93,<NA>
top,Abegondo,<NA>
freq,1,<NA>
mean,NaN,12132.473118
std,NaN,28528.028535
min,NaN,1004.0
25%,NaN,2979.0
50%,NaN,5195.0
75%,NaN,9064.0


In [164]:
municipios_readhtml["Population (2024)[20]"].sum()


np.int64(1128320)

## Paso 6 . Preguntas

* Los cinco concellos más poblados de la provincia de coruña son A coruña , Santiago, Ferrol , Narón y Oleiros los cuales coinciden con lo mostrado en la página web en la sección superior a la tabla extraida

In [165]:
municipios.sort_values("poblacion", ascending=False)[0:5]

,municipio,poblacion
29,A Coruña,249261
77,Santiago de Compostela,99536
35,Ferrol,64218
53,Narón,39285
57,Oleiros,38333


* Actualmente se cuenta con 43 concellos que tienen menor población que 5000 habitantes en coruña , lo cual si calculamos el porcentaje que eso representa de los 93 concellos existentes representa un 46.24%

In [166]:
menos5000=municipios[municipios["poblacion"] < 5000]["poblacion"].count()
menos5000

np.int64(43)

* Si calculamos el porcanteje de población de los 1128320 habitantes de coruña que viven en los concellos con menos de 5000 , nos encontramos que representan un 46.24% 
* Concluímos que la provincia de coruña es una población bastante concentrada en puntos claves , esparciendose de una forma más lenta en los demás concellos

In [167]:
porcentaje = (menos5000 / municipios.shape[0]) * 100

print(f"Porcentaje de municipios: {porcentaje:.2f}%")

Porcentaje de municipios: 46.24%


* Confirmamos la anterior conclusión mostrando el porcentaje que representa solo la suma de los habitantes totales de los tres concellos más poblados de la población total , el cual trata de un 36.60%

In [168]:
poblaciontotal3=municipios.sort_values("poblacion", ascending=False)["poblacion"][0:3].sum()
poblaciontotal3


np.int64(413015)

In [169]:
porcentaje3=(poblaciontotal3 / municipios["poblacion"].sum()) * 100
print(f"Porcentaje de la población en los 3 concellos con más habitantes: {porcentaje3:.2f}%")

Porcentaje de la población en los 3 concellos con más habitantes: 36.60%


* Para conseguir reunir una cantidad parecida a la mitad de la población empezando con los más poblados procedemos con los siguientes pasos :
    * Ordenamos la columna de modo descendente 
    * Creamos una nueva columna la cual acumule la suma de las poblaciones anteriores , es decir , el elemento 2 acumula la suma de su propio elemento + elemento 1(anterior)
    * Mostramos que al llegar al concello de carballo tendríamos ese resultado y de que concellos se tratan

* Con esto observamos que al llegar al concello número 18 ,ordenado de mayor a menor , representaríamos a una cantidad similar a la suma de la mitad de la población.

In [170]:
mitad_poblacion = municipios["poblacion"].sum() / 2
poblacionCalculada=mitad_poblacion -poblaciontotal3
poblacionCalculada #nos faltarían esta cantidad de habitantes para llegar a la mitad de la población de la provincia


np.float64(151145.0)

In [171]:
municipios_ordenados = municipios.sort_values(
    "poblacion", ascending=False
)

In [172]:
municipios_ordenados["poblacion_acumulada"] = municipios_ordenados["poblacion"].cumsum()

In [173]:
municipiosascumulados=municipios_ordenados[
    municipios_ordenados["poblacion_acumulada"] >= mitad_poblacion
].head(1)
municipiosascumulados

,municipio,poblacion,poblacion_acumulada
18,Carballo,31595,589078


In [174]:
municipios_necesarios = (
    municipios_ordenados["poblacion_acumulada"] < mitad_poblacion
).sum() + 1

municipios_necesarios
municipios_ordenados.head(municipios_necesarios)

,municipio,poblacion,poblacion_acumulada
29,A Coruña,249261,249261
77,Santiago de Compostela,99536,348797
35,Ferrol,64218,413015
53,Narón,39285,452300
57,Oleiros,38333,490633
4,Arteixo,34038,524671
1,Ames,32812,557483
18,Carballo,31595,589078


## Paso final . Guardar el resultado limpio en un csv

In [175]:
# Guardar datos limpios en csv
municipios.to_csv("../datos/clean/municipios_coruna_limpio.csv", index=False, sep=";", encoding="utf-8")

## Uso de IA
* Generar código de ejemplo para el calculo de municipios necesarios para llegar a la mitad de la población en el apartado de preguntas.
* Comparar o confirmar concluciones del apartado de preguntas.
* Generar guía a la hora de eliminar BOM del html a descargar.
* Se ha utilizado la herramienta chatgpt para uso de la IA en este ejercicio